# Novelty Search (Skeleton)
For use in the experiments of the Amorphous Fortress Narrative generation

### General Pseudocode
1. Create initial population of genomes
2. Evaluate each for fitness
3. Evaluate each for novelty against archive genomes
4. Add fit and novel genomes in archive
5. Select new parents of population from novelty archive
6. Mutate children and create new population
7. (Add random back in)
8. Repeat 2-7 for n generations

(Reference: [SimSim](https://github.com/lsoros/simsim/blob/master/simsim.cpp) and [Algorithm Definition](https://algorithmafternoon.com/novelty/novelty_search_algorithm/))

### Setup

In [1]:
# imports
import random
import numpy as np
import json
import re

In [2]:
# import data
ALL_SUBJS = np.load('../bank_files/cn_full_subjects.npy', allow_pickle=True)
ALL_OBJS = np.load('../bank_files/cn_full_objects.npy', allow_pickle=True)
ALL_VERBS = np.load('../bank_files/cn_full_verbs.npy', allow_pickle=True) 
CN_GRAPH = json.load(open('../bank_files/full_word_graph_noweight.json'))

# convert to normal lists
ALL_SUBJS = [str(s) for s in ALL_SUBJS]
ALL_OBJS = [str(o) for o in ALL_OBJS]
ALL_VERBS = [str(v) for v in ALL_VERBS]

print(len(ALL_SUBJS), len(ALL_OBJS), len(ALL_VERBS))

3701 3377 1389


In [21]:
# constants
MC_MUTATE_PERC = 0.25
ENT_MUTATE_PERC = 0.1
VERB_MUTATE_PERC = 0.25
POP_SIZE = 20
NUM_GENERATIONS = 100

AF_VERBS = ["moved", "died", "cloned", "took", "pushed", "added", "transformed", "blocked", "chased"]

In [ ]:
# maps a log for usage in the novelty search
class AF_Story:
    def __init__(self, log_file):
        with open(log_file, 'r') as f:
            self.og_text = [line.strip() for line in f.readlines()]
        self.ent_ids = self.find_spec_ents()        # dict of entities with subject/object designation
        self.ent_reps = self.get_ent_reps()

        self.ent_order, self.mc_ent = self.find_ents()      # list of all entities in the original text
        self.verb_set = self.find_verbs()          # list of tuples of (subject entity, verb)
        self.verb_order = [v[1] for v in self.verb_set.values()]    # list of all verbs in the original text


    def find_ents(self):
        ''' Find all AF entities in the original text. '''
        af_ents = []
        for line in self.og_text:
            match = re.findall(r'(\[.\..{4}\])', line)
            if match:
                af_ents.extend(match)

        # get highest occuring entity as main character
        random.shuffle(af_ents) # shuffle to avoid biasing first entity as MC
        mc_ent = max(set(af_ents), key = af_ents.count)
        return list(set(af_ents)), mc_ent

    def find_spec_ents(self):
        ''' Identifies entities and whether they are the subject or object in the sentence. '''
        af_ents = {}
        for line in self.og_text:
            match = re.findall(r'(\[.\..{4}\])', line)
            if match:
                for i in range(len(match)):
                    if i == 0:
                        af_ents[match[i]] = 'subject'
                    elif match[i] not in af_ents:
                        af_ents[match[i]] = 'object'
                    
        return af_ents
    
    def get_ent_reps(self):
        ''' Get the symbol representations of the entities in the story '''
        return list(set([e[1] for e in self.ent_ids.keys()]))
    
    def find_verbs(self):
        ''' Find all verbs in the original text '''
        af_verbs = {}
        for i, line in enumerate(self.og_text):
            subj_ent = re.findall(r'(\[.\..{4}\])', line)
            for verb in AF_VERBS:
                if verb in line:
                    af_verbs[i] = (subj_ent[0], verb)
                    break # only take the first verb found
        return af_verbs

In [23]:
# test
stupid_story = AF_Story('../logs/stupid_log.txt')
print("Ent IDs:\t" + str(stupid_story.ent_ids))
print("Ent Reps:\t" + str(stupid_story.ent_reps))
print("MC Ent:\t" + str(stupid_story.mc_ent))
print("Ent Order:\t" + str(stupid_story.ent_order))
print("Verb Set:\t" + str(stupid_story.verb_set))
print("Verb Order:\t" + str(stupid_story.verb_order))

Ent IDs:	{'[%.b29d]': 'subject', '[y.c9ae]': 'subject', '[ .d474]': 'subject', '[ .6adb]': 'object', '[$.b14a]': 'subject', '[*.7b93]': 'subject', '[|.c91d]': 'subject', '[}.460f]': 'subject', '[5.6b26]': 'object', '[S.d487]': 'subject', '[!.fdaf]': 'object', '[Y.3a62]': 'object', '[o.8829]': 'object', '[;.f34e]': 'subject', '[9.547d]': 'subject'}
Ent Reps:	['y', '|', '5', '*', ' ', '9', '}', 'Y', 'S', '%', '!', '$', ';', 'o']
MC Ent:	[$.b14a]
Ent Order:	['[$.b14a]', '[y.c9ae]', '[9.547d]', '[S.d487]', '[ .d474]', '[Y.3a62]', '[5.6b26]', '[o.8829]', '[%.b29d]', '[}.460f]', '[|.c91d]', '[;.f34e]', '[*.7b93]', '[ .6adb]', '[!.fdaf]']
Verb Set:	{3: ('[%.b29d]', 'transformed'), 4: ('[ .d474]', 'cloned'), 5: ('[$.b14a]', 'moved'), 6: ('[*.7b93]', 'moved'), 7: ('[ .d474]', 'transformed'), 8: ('[}.460f]', 'blocked'), 9: ('[S.d487]', 'added'), 10: ('[$.b14a]', 'moved'), 11: ('[|.c91d]', 'pushed'), 12: ('[y.c9ae]', 'chased'), 13: ('[;.f34e]', 'added'), 14: ('[9.547d]', 'pushed')}
Verb Order:	['

In [ ]:
# Object to store the genome info
class FicGenome:
    def __init__(self, mc:str=None, ent:dict=None, verbs:dict=None):
        '''
            mc:    str
            ent:   {og_story_id: fic_rep_ent}
            verbs: {line: fic_rep_verb}
        '''
        self.mc = mc
        self.ent = ent
        self.verbs = verbs
        self.fitness = 0
        self.genome = None

    def clone(self):
        ''' Returns a separate copy of this object '''
        new_fic = FicGenome(self.mc, {k:v for k,v in self.ent.items()}, {k:v for k,v in self.verbs.items()})
        new_fic.fitness = self.fitness
        new_fic.genome = self.genome
        return new_fic

    def mutate(self, mc='random', ent='random', verbs='random'):
        ''' Mutates the FicGenome object
            mc:    'random' | 'same'
            ent:   'random' | 'assoc'
            verbs: 'random' | 'assoc'
        '''
        # TODO: mutate based on associations
        pass

    def eval(self):
        ''' Evaluates the fitness of the FicGenome object '''
        pass

    def make_genome(self):
        ''' Creates a representation of the genome (ents+verbs) '''
        pass


    def generate_story(self, af_story, out_file:str=None):
        ''' Creates a story log based on the genome '''
        new_story = []
        for i, og_line in enumerate(af_story.og_text):
            new_line = og_line
            
            # Replace entities in the original line with their representations
            for ent_id, fic_rep in self.ent.items():
                new_line = new_line.replace(ent_id, fic_rep)

            # Replace verbs in the original line with their representations
            if i in self.verbs:
                af_story_verb = af_story.verb_set[i]
                fic_verb = self.verbs[i]
                new_line = new_line.replace(af_story_verb[1], fic_verb)

            new_story.append(new_line)

        # Write the modified line to the output file
        if out_file is not None:
            with open(out_file, 'w') as f:
                for line in new_story:
                    f.write(line + '\n')

        return new_story

### Perform novelty search on a stripped log

In [29]:
def init_population(size, story, style='random'):
    ''' Initializes a population of FicGenome objects based on the given AF_Story object '''
    population = []

    if style == 'random':
        # initialize fully randomly
        for _ in range(size):
            # choose a random MC from the entities in the story
            mc = random.choice(ALL_SUBJS)
            
            # choose random entities from the entities in the story
            ent = {}
            for k,v in story.ent_ids.items():
                if k == story.mc_ent:
                    ent[story.mc_ent] = mc
                else:
                    if v == 'subject':
                        ent[k] = random.choice(ALL_SUBJS)
                    else:
                        ent[k] = random.choice(ALL_OBJS)

            # choose completely random verbs from the verbs in the story
            verbs = {}
            for k,v in story.verb_set.items():
                # pick random verb
                verbs[k] = random.choice(ALL_VERBS)
                

            genome = FicGenome(mc, ent, verbs)
            population.append(genome)

    # TODO: create population based on associations

    
    return population

In [30]:
# test generating a story
pop = init_population(5, stupid_story, style='random')
for i, fic in enumerate(pop):
    print(f"\nFic {i}: MC={fic.mc}, Ents={fic.ent}, Verbs={fic.verbs}")
    fic.generate_story(stupid_story, out_file=f'../gen_stories/stupid_fic_{i}.txt')


Fic 0: MC=industry, Ents={'[%.b29d]': 'wave', '[y.c9ae]': 'invitation', '[ .d474]': 'contribution', '[ .6adb]': 'earthquake', '[$.b14a]': 'industry', '[*.7b93]': 'mask', '[|.c91d]': 'poker', '[}.460f]': 'verification', '[5.6b26]': 'deer', '[S.d487]': 'listener', '[!.fdaf]': 'viola', '[Y.3a62]': 'aim', '[o.8829]': 'penut', '[;.f34e]': 'accordion', '[9.547d]': 'radiation'}, Verbs={3: 'firehose', 4: 'economize', 5: 'record', 6: 'lead', 7: 'sell', 8: 'drain', 9: 'unworn', 10: 'deploy', 11: 'hand', 12: 'answer', 13: 'shift', 14: 'elect'}

Fic 1: MC=shopping, Ents={'[%.b29d]': 'habit', '[y.c9ae]': 'merger', '[ .d474]': 'bead', '[ .6adb]': 'shrine', '[$.b14a]': 'shopping', '[*.7b93]': 'department', '[|.c91d]': 'corner', '[}.460f]': 'tounge', '[5.6b26]': 'england', '[S.d487]': 'flatmate', '[!.fdaf]': 'appointment', '[Y.3a62]': 'pearl', '[o.8829]': 'assignment', '[;.f34e]': 'penis', '[9.547d]': 'mustache'}, Verbs={3: 'milk', 4: 'contribute', 5: 'accrue', 6: 'stimuli', 7: 'leak', 8: 'map', 9: '

In [12]:
def is_novel(x, archive, threshold=0.5):
    ''' Determines if a FicGenome object is novel compared to an archive of FicGenome objects '''
    # todo compare vector distance of genomes
    return True

In [18]:
def novelty_search(af_log, fit_threshold=0.5, novel_threshold=0.5, rand_perc=0.2):
    ''' Main novelty search algorithm '''

    # 0. Initialize story representation
    story = AF_Story(af_log)
    
    # 1. Initialize population and archive
    population = init_population(POP_SIZE, story)
    archive = []

    for gen in range(NUM_GENERATIONS):
        print(f"Generation {gen}")

        # 8. Repeat 2-7 for NUM_GENERATIONS
        for indiv in population:

            # 2. Evaluate fitness
            indiv.eval()

            # 3+4. Evaluate novelty against archive and add if novel and fit enough
            if is_novel(indiv, archive, novel_threshold) and indiv.fitness > fit_threshold:
                archive.append(indiv.clone())


        # print some stats
        population.sort(key=lambda x: x.fitness, reverse=True)
        fit_scores = [indiv.fitness for indiv in population]
        print(f"  Pop Fitness: max {max(fit_scores):.3f}, min {min(fit_scores):.3f}, avg {sum(fit_scores)/len(fit_scores):.3f}")
        print(f"  Archive size: {len(archive)}")

        if gen % 10 == 0:
            print("   FicGenome of best population individual:")
            print(f"     - Best MC: {population[0].mc}")
            print(f"     - Best Ents: {list(population[0].ent.values())}")
            print(f"     - Best Verbs: {set(population[0].verbs)}")

        

        

        # 5. Select new parents from novelty archive
        if len(archive) > 0:
            parents = random.choices(archive, k=int(POP_SIZE*(1-rand_perc)))
        else:
            parents = random.choices(population, k=int(POP_SIZE*(1-rand_perc)))

        # 6. Mutate children from parents
        new_pop = []
        for parent in parents:
            child = parent.clone()
            child.mutate()
            new_pop.append(child)

        # 7. Add random individuals
        rand_amt = (POP_SIZE - len(new_pop))
        for _ in range(int(POP_SIZE*rand_perc)):
            randos = init_population(rand_amt, story)
            new_pop.extend(randos)

        # update population
        population = new_pop

    return archive

In [19]:
arc = novelty_search('../logs/stupid_log.txt', fit_threshold=0.5, novel_threshold=0.5, rand_perc=0.2)

Generation 0
  Pop Fitness: max 0.000, min 0.000, avg 0.000
  Archive size: 0
   FicGenome of best population individual:
     - Best MC: nursing
     - Best Ents: ['heating', 'furniture', 'toy', 'wash', 'nursing', 'album', 'flash', 'turttle', 'television', 'duster', 'number', 'nut', 'entry', 'gardener', 'paddler']
     - Best Verbs: {3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14}
Generation 1
  Pop Fitness: max 0.000, min 0.000, avg 0.000
  Archive size: 0
Generation 2
  Pop Fitness: max 0.000, min 0.000, avg 0.000
  Archive size: 0
Generation 3
  Pop Fitness: max 0.000, min 0.000, avg 0.000
  Archive size: 0
Generation 4
  Pop Fitness: max 0.000, min 0.000, avg 0.000
  Archive size: 0
Generation 5
  Pop Fitness: max 0.000, min 0.000, avg 0.000
  Archive size: 0
Generation 6
  Pop Fitness: max 0.000, min 0.000, avg 0.000
  Archive size: 0
Generation 7
  Pop Fitness: max 0.000, min 0.000, avg 0.000
  Archive size: 0
Generation 8
  Pop Fitness: max 0.000, min 0.000, avg 0.000
  Archive size: 